# Databricks Dynamic Ingestion Engine - Step 01: Ingestion Pipeline
This notebook serves as the generic, schemaless dynamic ingestion pipeline.
Given a `target_table_name` widget parameter, it fetches metadata configuration from `bronze_db.ingestion_config`, dynamically validates each row using the **Pydantic** validation engine, writes valid records to `silver_db.<target_table_name>`, writes invalid records to `bronze_db.quarantine_<target_table_name>`, and records execution metrics in `bronze_db.ingestion_audit`.



In [0]:
single_table = "supplies"
# Widget Setup for Parameterized Execution
try:
    dbutils.widgets.text("target_table_name", f"{single_table}", "Target Table Name")
    target_table_name = dbutils.widgets.get("target_table_name")
except Exception:
    target_table_name = single_table

print(f"Executing Dynamic Ingestion for Target Table: {target_table_name}")



In [0]:
import uuid
import json
import datetime
import urllib.request
import os
from pydantic import create_model, ValidationError
from typing import Optional, Any
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType, IntegerType, FloatType, BooleanType

# Step 1: Read Ingestion Configuration for the Target Table
config_df = spark.sql(f"""
    SELECT * FROM bronze_db.ingestion_config 
    WHERE target_table_name = '{target_table_name}'
    ORDER BY col_seq_no ASC
""").collect()

if not config_df:
    raise ValueError(f"No metadata configuration found in bronze_db.ingestion_config for target_table_name='{target_table_name}'")

# Extract metadata variables
source_location = config_df[0]['source_location']
bronze_location = config_df[0]['bronze_location']
target_table_db = config_df[0]['target_table_db']
target_table_name = config_df[0]['target_table_name']

print(f"Source URL: {source_location}")
print(f"Bronze Path: {bronze_location}")
print(f"Target DB: {target_table_db}.{target_table_name}")



In [0]:
# Step 2: Ensure Data Availability in Bronze Storage
# Download source file from source_location to bronze_location if needed
local_bronze_dir = os.path.dirname(bronze_location)
os.makedirs(local_bronze_dir, exist_ok=True)

if not os.path.exists(bronze_location):
    print(f"Downloading dataset from {source_location} to {bronze_location}...")
    urllib.request.urlretrieve(source_location, bronze_location)
    print("Download complete.")

# Read Raw CSV via pandas (serverless Spark cannot read local /tmp paths)
import pandas as pd
df_pandas = pd.read_csv(bronze_location, dtype=str)
df_raw = spark.createDataFrame(df_pandas)
total_records_processed = df_raw.count()
print(f"Total raw records loaded: {total_records_processed}")



In [0]:
# Step 3: Pydantic Data Quality Validation & Casting Engine
job_id = str(uuid.uuid4())
start_time = datetime.datetime.now(datetime.timezone.utc)

# Build Dynamic Pydantic Validation Model based on Metadata Rules
def build_pydantic_model(model_name: str, cols_config: list):
    fields = {}
    for col in cols_config:
        t_name = col['target_column_name']
        t_type = col['target_datatype']
        pattern = col['target_data_pattern'] or ''
        
        if t_type == 'integer':
            py_type = int
        elif t_type == 'float':
            py_type = float
        elif t_type == 'boolean':
            py_type = bool
        elif t_type == 'timestamp':
            py_type = datetime.datetime
        else:
            py_type = str
            
        is_required = 'NOT_NULL' in pattern
        if not is_required:
            py_type = Optional[py_type]
            default_val = None
        else:
            default_val = ...
            
        fields[t_name] = (py_type, default_val)
        
    return create_model(model_name, **fields)

DynamicModel = build_pydantic_model(f"{target_table_name.capitalize()}Validator", config_df)

# Prepare mapping dicts
col_mapping = {row['source_column_name']: row['target_column_name'] for row in config_df}
col_types = {row['target_column_name']: (row['target_datatype'], row['target_data_pattern'] or '') for row in config_df}

raw_rows = [row.asDict() for row in df_raw.collect()]

valid_rows = []
quarantine_rows = []

for idx, r in enumerate(raw_rows):
    # Rename keys from source_column_name to target_column_name & type conversion
    converted_dict = {}
    validation_error = None
    
    for src_col, tgt_col in col_mapping.items():
        raw_val = r.get(src_col)
        t_type, pattern = col_types[tgt_col]
        
        if raw_val is None or raw_val.strip() == '':
            converted_dict[tgt_col] = None
        else:
            try:
                if t_type == 'integer':
                    converted_dict[tgt_col] = int(raw_val)
                    if '>=0' in pattern and converted_dict[tgt_col] < 0:
                        raise ValueError(f"Value {converted_dict[tgt_col]} must be >= 0")
                elif t_type == 'float':
                    converted_dict[tgt_col] = float(raw_val)
                    if '>=0' in pattern and converted_dict[tgt_col] < 0:
                        raise ValueError(f"Value {converted_dict[tgt_col]} must be >= 0")
                elif t_type == 'boolean':
                    val_str = str(raw_val).strip().lower()
                    if val_str in ['true', '1', 't', 'yes']:
                        converted_dict[tgt_col] = True
                    elif val_str in ['false', '0', 'f', 'no']:
                        converted_dict[tgt_col] = False
                    else:
                        raise ValueError(f"Invalid boolean string: {raw_val}")
                elif t_type == 'timestamp':
                    # Handle standard ISO formats
                    clean_ts = raw_val.replace('Z', '')
                    converted_dict[tgt_col] = datetime.datetime.fromisoformat(clean_ts)
                else:
                    converted_dict[tgt_col] = str(raw_val)
            except Exception as e:
                validation_error = f"Column '{tgt_col}' conversion error: {str(e)}"
                break

    # Pydantic schema validation check
    if not validation_error:
        try:
            validated_obj = DynamicModel(**converted_dict)
            valid_record = validated_obj.model_dump()
            valid_record['ingestion_timestamp'] = start_time
            valid_rows.append(valid_record)
        except ValidationError as val_err:
            validation_error = f"Pydantic Validation Error: {str(val_err)}"
            
    if validation_error:
        quarantine_rows.append({
            "quarantine_id": str(uuid.uuid4()),
            "job_id": job_id,
            "quarantine_timestamp": start_time,
            "raw_data_json": json.dumps(r),
            "error_message": validation_error
        })

print(f"Validation Completed: {len(valid_rows)} Valid Records, {len(quarantine_rows)} Quarantined Records")



In [0]:
# Step 4: Write Clean Data to Silver Delta Table
if valid_rows:
    df_valid = spark.createDataFrame(valid_rows)
    df_valid.write.format("delta").mode("overwrite").saveAsTable(f"{target_table_db}.{target_table_name}")
    print(f"Successfully saved {len(valid_rows)} records to {target_table_db}.{target_table_name}")
else:
    print(f"No valid records to write to {target_table_db}.{target_table_name}")

# Step 5: Write Quarantined Records to Quarantine Delta Table
quarantine_table_name = f"quarantine_{target_table_name}"
if quarantine_rows:
    df_quarantine = spark.createDataFrame(quarantine_rows)
    df_quarantine.write.format("delta").mode("overwrite").saveAsTable(f"bronze_db.{quarantine_table_name}")
    print(f"Successfully saved {len(quarantine_rows)} quarantined records to bronze_db.{quarantine_table_name}")
else:
    # Ensure quarantine table exists even if empty
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS bronze_db.{quarantine_table_name} (
            quarantine_id STRING,
            job_id STRING,
            quarantine_timestamp TIMESTAMP,
            raw_data_json STRING,
            error_message STRING
        ) USING DELTA
    """)
    print(f"Quarantine table bronze_db.{quarantine_table_name} verified.")



In [0]:
# Step 6: Log Execution Metrics into bronze_db.ingestion_audit
end_time = datetime.datetime.now(datetime.timezone.utc)
status = "SUCCESS" if len(quarantine_rows) == 0 else ("PARTIAL_SUCCESS" if len(valid_rows) > 0 else "FAILED")
error_summary = f"{len(quarantine_rows)} records failed quality checks" if len(quarantine_rows) > 0 else None

audit_schema = StructType([
    StructField("job_id", StringType(), True),
    StructField("target_table_db", StringType(), True),
    StructField("target_table_name", StringType(), True),
    StructField("source_location", StringType(), True),
    StructField("bronze_location", StringType(), True),
    StructField("start_time", TimestampType(), True),
    StructField("end_time", TimestampType(), True),
    StructField("total_records_processed", LongType(), True),
    StructField("success_count", LongType(), True),
    StructField("failed_count", LongType(), True),
    StructField("status", StringType(), True),
    StructField("error_summary", StringType(), True)
])

audit_record = [(
    job_id,
    target_table_db,
    target_table_name,
    source_location,
    bronze_location,
    start_time,
    end_time,
    int(total_records_processed),
    len(valid_rows),
    len(quarantine_rows),
    status,
    error_summary
)]

df_audit = spark.createDataFrame(audit_record, audit_schema)
df_audit.write.format("delta").mode("append").saveAsTable("bronze_db.ingestion_audit")

print("Ingestion Job Execution Audit logged successfully:")
display(df_audit)



In [0]:
dbutils.widgets.remove("target_table_name")
print("Widget 'target_table_name' successfully removed.")